══════════════════════════════════════════════════════════════
 WEEK 11  |  DAY 5  |  AI-POWERED DATA ASSISTANT
══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Build a data analysis module that answers questions about a dataset
 2. Train a churn prediction model and run fairness checks
 3. Classify support tickets and answer policy questions
 4. Combine all modules into a single assistant router
 5. Evaluate AI pipeline output systematically (assert-based + LLM-as-judge)

 TIME:  ~75 minutes

 YOUTUBE
 ───────
 Search: "Python build AI assistant tutorial"
 Search: "customer churn prediction scikit-learn"
 Search: "LLM evaluation testing AI applications Python"

 NOTE: This lesson combines skills from Weeks 9, 10, and 11:
       ML model (Week 9), LLM API + RAG (Week 10), NLP + Ethics (Week 11)

══════════════════════════════════════════════════════════════

In [ ]:

import os
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score




 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: RAGAS automated metrics vs LLM-as-judge (GPT-4/Claude) vs human annotation.
 Rule of thumb: use RAGAS for continuous integration checks on RAG pipelines. Use LLM-as-judge for fast qualitative comparison of prompt variants. Use human annotation when launching or when automated scores diverge from user satisfaction.

══════════════════════════════════════════════════════════════
 CONCEPT 1 — DATA ANALYSIS MODULE
══════════════════════════════════════════════════════════════

The first component of any data assistant is the ability to answer
questions about a dataset using pandas. This module takes a customer
DataFrame and prints key metrics: totals, averages, and breakdowns
by category (city, churn status, etc.).


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print("=" * 60)
print("CONCEPT 1: Data Analysis Module")
print("=" * 60)
print()



Generate synthetic retail dataset


In [ ]:
np.random.seed(42)
n = 500

customers = pd.DataFrame({
    "customer_id":      range(1001, 1001 + n),
    "age":              np.random.randint(18, 70, n),
    "gender":           np.random.choice(["Male", "Female"], n),
    "city":             np.random.choice(["Tel Aviv", "Jerusalem", "Haifa", "Beer Sheva"], n),
    "monthly_spend":    np.round(np.random.uniform(100, 5000, n), 2),
    "num_orders":       np.random.randint(1, 50, n),
    "num_complaints":   np.random.randint(0, 10, n),
    "months_active":    np.random.randint(1, 60, n),
    "churned":          np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

print(f"Dataset: {customers.shape[0]} customers, {customers.shape[1]} columns")
print(customers.head())

def analyze_customers(df):
    """Basic customer analysis."""
    print(f"\nTotal customers:       {len(df):,}")
    print(f"Average monthly spend: ${df['monthly_spend'].mean():,.2f}")
    print(f"Churn rate:            {df['churned'].mean():.1%}")
    print()
    print("Average spend by city:")
    print(df.groupby("city")["monthly_spend"].mean().sort_values(ascending=False).round(2))
    print()
    print("Churn rate by city:")
    print(df.groupby("city")["churned"].mean().sort_values(ascending=False).map("{:.1%}".format))

analyze_customers(customers)




══════════════════════════════════════════════════════════════
 EXERCISE 1 — Extended Customer Analysis
══════════════════════════════════════════════════════════════

Add to the analysis:
  1. Average number of complaints for churned vs non-churned customers
     (churned customers likely complain more — verify this)
  2. The city with the highest churn rate
  3. Customers who are "high risk": num_complaints > 5 AND months_active < 12
     Print how many high-risk customers there are

Expected output:
    Avg complaints (churned):     X.X
    Avg complaints (not churned): X.X
    City with highest churn: <city_name>
    High-risk customers: XX


══════════════════════════════════════════════════════════════
 CONCEPT 2 — CHURN PREDICTION MODEL
══════════════════════════════════════════════════════════════

A DecisionTree classifier predicts whether a customer will churn
based on their behavior features. After training, we wrap it in a
function that accepts a single customer dict and returns a risk level.


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print()
print("=" * 60)
print("CONCEPT 2: Churn Prediction Model")
print("=" * 60)

features = ["age", "monthly_spend", "num_orders", "num_complaints", "months_active"]
X = customers[features]
y = customers["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

churn_model = DecisionTreeClassifier(max_depth=4, random_state=42)
churn_model.fit(X_train, y_train)
acc = accuracy_score(y_test, churn_model.predict(X_test))
print(f"Churn model accuracy: {acc:.1%}")

def predict_churn(customer_data):
    """Predict churn risk for a single customer."""
    df = pd.DataFrame([customer_data])[features]
    proba = churn_model.predict_proba(df)[0]
    return {
        "churn_risk":    "HIGH" if proba[1] > 0.6 else "MEDIUM" if proba[1] > 0.3 else "LOW",
        "churn_proba":   f"{proba[1]:.1%}",
        "stay_proba":    f"{proba[0]:.1%}",
    }



Test the prediction function


In [ ]:
new_customer = {
    "age": 35, "monthly_spend": 800, "num_orders": 5,
    "num_complaints": 8, "months_active": 6
}
print()
print("Prediction for test customer:")
print(f"  Input: {new_customer}")
print(f"  Result: {predict_churn(new_customer)}")




══════════════════════════════════════════════════════════════
 EXERCISE 2 — Churn Predictions and Fairness Check
══════════════════════════════════════════════════════════════

1. Run predict_churn() on 5 different customers with varying profiles:
   - A loyal customer (high orders, low complaints, long active)
   - A new unhappy customer (low orders, high complaints, short active)
   - Print the risk and probability for each

2. Fairness check: does the model predict churn at different rates
   for males vs females?
   Use customers.groupby("gender") to calculate average predicted
   churn probability. Flag if gap > 5%.

Expected output:
    Customer 1 (loyal):   LOW  - churn probability: X%
    Customer 2 (unhappy): HIGH - churn probability: X%
    ...
    Gender fairness check:
      Male avg churn prob:   XX%
      Female avg churn prob: XX%
      Gap: X% — OK / WARNING


══════════════════════════════════════════════════════════════
 CONCEPT 3 — SUPPORT TICKETS & KNOWLEDGE BASE Q&A
══════════════════════════════════════════════════════════════

Two more assistant capabilities:
  1. Classify support tickets into categories (billing, technical, etc.)
  2. Answer policy questions from a knowledge base (simple RAG pattern)

In production, both would use an LLM. Here we use rule-based fallbacks
that work without an API key.


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print()
print("=" * 60)
print("CONCEPT 3: Support Tickets & Knowledge Base")
print("=" * 60)



--- Ticket classifier ---


In [ ]:
tickets = [
    "I was charged twice this month for my subscription.",
    "The app crashes every time I try to open my account.",
    "My order from 2 weeks ago still hasn't arrived.",
    "How do I change my password?",
    "I received a damaged product in my delivery.",
]

def classify_ticket(ticket_text):
    """
    Classify a support ticket into a category.
    Categories: billing | technical | shipping | account | other
    """
    t = ticket_text.lower()
    if any(w in t for w in ["charged", "payment", "subscription", "invoice", "refund"]):
        return "billing"
    if any(w in t for w in ["crash", "error", "bug", "app", "login", "slow"]):
        return "technical"
    if any(w in t for w in ["delivery", "shipped", "arrived", "order", "package"]):
        return "shipping"
    if any(w in t for w in ["password", "account", "email", "username"]):
        return "account"
    return "other"

print()
print(f"{'Ticket':55} {'Category'}")
print("-" * 70)
for ticket in tickets:
    category = classify_ticket(ticket)
    print(f"{ticket[:55]:55} {category}")



--- Knowledge base Q&A ---


In [ ]:
policies = {
    "refund_001":  "Customers can request a full refund within 30 days of purchase. "
                   "Refund requests must be submitted through the app or by email.",
    "shipping_001":"Standard delivery takes 3-5 business days. Express delivery (1-2 days) "
                   "is available for an additional fee. Free shipping on orders over 200 NIS.",
    "account_001": "Passwords must be at least 8 characters and include a number. "
                   "Accounts are locked after 5 failed login attempts. "
                   "To unlock, use 'forgot password' or contact support.",
    "warranty_001":"Products carry a 12-month manufacturer warranty. "
                   "Warranty covers manufacturing defects but not physical damage.",
}

def answer_policy_question(question):
    """Simple keyword-based policy retrieval (replace with ChromaDB for production)."""
    question_lower = question.lower()
    best_match = None
    for policy_id, text in policies.items():
        if any(word in text.lower() for word in question_lower.split()):
            best_match = text
            break
    if best_match:
        return best_match
    return "I don't have information about that. Please contact support@company.com"

print()
policy_questions = [
    "How long do I have to return a product?",
    "What is the password policy?",
    "How long does delivery take?",
]

for q in policy_questions:
    answer = answer_policy_question(q)
    print(f"Q: {q}")
    print(f"A: {answer[:120]}...")
    print()




══════════════════════════════════════════════════════════════
 EXERCISE 3 — Build the Complete Assistant Router
══════════════════════════════════════════════════════════════

Write a function called data_assistant(user_input) that:
  1. Detects the type of request from the user_input text:
       "churn" / "predict"              -> call predict_churn() with sample data
       "ticket" / "support"             -> call classify_ticket()
       "policy" / "shipping" / "refund" -> call answer_policy_question()
       "stats" / "analysis"             -> call analyze_customers()
       else -> "I can help with: churn prediction, ticket classification,
                policy questions, data analysis"

  2. Routes the request to the right function
  3. Returns a helpful response string

Test with:
  data_assistant("Can you predict if this customer will churn?")
  data_assistant("Classify this support ticket: I was charged twice")
  data_assistant("What is the refund policy?")
  data_assistant("Show me customer statistics")
  data_assistant("What is the weather today?")

Expected output:
    Request: "Can you predict if this customer will churn?"
    Response: churn_risk=HIGH, churn_proba=65% ...

    Request: "What is the weather today?"
    Response: I can help with: churn prediction, ticket classification, ...


══════════════════════════════════════════════════════════════
 CONCEPT 4 — LLM EVALUATION
══════════════════════════════════════════════════════════════

You cannot ship an AI application without knowing if it works.
LLM output is non-deterministic — the same prompt can give different
answers each time. You need a systematic way to test quality.

THREE EVALUATION APPROACHES:

1. ASSERT-BASED (no API needed — fast and cheap)
   Test your pipeline functions against known correct answers.
   Check: does the output contain the right keyword? the right format?
   Use this for classifiers, extractors, and structured outputs.

2. LLM-AS-JUDGE (requires API — for open-ended answers)
   Send the original question + the generated answer to a second LLM.
   Ask it: "Rate this answer 1-5 for correctness. Return JSON."
   Useful when there is no single correct answer to check against.

3. RAGAS (for RAG pipelines — specialized library)
   Measures three things automatically:
     faithfulness:       does the answer match the retrieved context?
     answer_relevancy:   does the answer address the question?
     context_precision:  was the right context retrieved?
   Install: pip install ragas

WHAT TO TEST IN AN AI PIPELINE:
  - Classifier accuracy (assert-based)
  - Structured output format (does it parse? are fields present?)
  - Response relevance (LLM-as-judge)
  - Edge cases: empty input, unexpected language, very long text
  - Regression: did a prompt change break existing test cases?

RULE: if you change a prompt, re-run all tests.
A small wording change can silently break output format or classification.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 60)
print("CONCEPT 4: LLM Evaluation")
print("=" * 60)

# Assert-based evaluation — works without any API key
# Test each classifier case and track pass/fail

evaluation_cases = [
    ("I was charged twice for my subscription.",   "billing"),
    ("The app crashes on startup.",                "technical"),
    ("My package hasn't arrived after 2 weeks.",  "shipping"),
    ("How do I reset my password?",               "account"),
    ("I love your product!",                      "other"),
]

def evaluate_classifier(classifier_fn, test_cases):
    """Evaluate a classifier function against labeled test cases."""
    passed = 0
    failures = []
    for text, expected in test_cases:
        predicted = classifier_fn(text)
        if predicted == expected:
            passed += 1
        else:
            failures.append({"input": text, "expected": expected, "got": predicted})
    total    = len(test_cases)
    accuracy = passed / total
    return {"passed": passed, "total": total, "accuracy": accuracy, "failures": failures}

result = evaluate_classifier(classify_ticket, evaluation_cases)
print(f"Ticket classifier accuracy: {result['accuracy']:.1%}  ({result['passed']}/{result['total']})")
if result["failures"]:
    print("Failures:")
    for f in result["failures"]:
        print(f"  Input:    {f['input'][:55]}")
        print(f"  Expected: {f['expected']}  Got: {f['got']}")
else:
    print("All test cases passed.")

print()
print("LLM-as-judge pattern (requires API key):")
print("  For open-ended answers that can't be exact-matched,")
print("  send the question + answer to a second LLM and ask:")
print("  'Rate this answer 1-5 for correctness and relevance.'")
print("  Use model='gpt-4o-mini' for the judge to keep costs low.")


══════════════════════════════════════════════════════════════
 EXERCISE 4 — Evaluate the Policy Q&A Module
══════════════════════════════════════════════════════════════
Write a function evaluate_policy_qa(qa_fn, test_cases) that:
  - Takes a Q&A function and a list of (question, must_contain) pairs
  - For each case: calls qa_fn(question) and checks if the answer
    contains must_contain (case-insensitive)
  - Tracks how many passed and which ones failed

Run it against answer_policy_question() using qa_test_cases below.
Print the accuracy and details of any failures.

Then fix any failing case: either update the policies dict or the
answer_policy_question() logic so all 4 tests pass.

Expected output:
    Evaluation results:
    Passed: X/4
    Accuracy: XX%

    Failures (if any):
      Q: "Is there free shipping?"
      Expected keyword: "200 NIS"
      Got: "I don't have information..."

    After fix:
    Passed: 4/4  Accuracy: 100%

In [ ]:

qa_test_cases = [
    ("How long do I have to return a product?", "30 days"),
    ("What is the password policy?",            "8 characters"),
    ("How long does delivery take?",            "business days"),
    ("Is there free shipping?",                 "200 NIS"),
]






══════════════════════════════════════════════════════════════
 CONCEPT 5 — A/B TESTING FOR LLMS: DATA-DRIVEN PROMPT OPTIMIZATION
══════════════════════════════════════════════════════════════
 Changing a prompt without measuring the impact is guessing.
 A/B testing gives you evidence: prompt A produces better results than B.

 A/B test structure:
   Control (A)   : the current prompt in production
   Treatment (B) : the new prompt you want to test
   Test inputs   : a representative sample of real user inputs
   Scorer        : a function that rates each response (0.0 to 1.0)

 Scorer types:
   Automated : length, keyword presence, format check (fast, no cost)
   LLM judge : ask a second LLM to rate the response (accurate, costs more)
   Human eval: human rates each response (ground truth, expensive)

 Winner selection: mean(scores_B) > mean(scores_A) → promote B

 Production workflow:
   1. Define scorer aligned with your quality definition
   2. Run A/B on 50-100 representative inputs
   3. Promote winner, archive loser, document the decision

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import time


def run_ab_test(prompt_a: str, prompt_b: str,
                test_inputs: list[str],
                llm_fn,
                scorer) -> dict:
    """
    Run a head-to-head comparison of two prompts.
    llm_fn(prompt) -> str
    scorer(input, response) -> float  (0.0 = bad, 1.0 = perfect)
    """
    scores_a, scores_b = [], []

    for user_input in test_inputs:
        full_a = prompt_a.format(input=user_input)
        full_b = prompt_b.format(input=user_input)

        resp_a = llm_fn(full_a)
        resp_b = llm_fn(full_b)

        score_a = scorer(user_input, resp_a)
        score_b = scorer(user_input, resp_b)

        scores_a.append(score_a)
        scores_b.append(score_b)

    avg_a = sum(scores_a) / len(scores_a)
    avg_b = sum(scores_b) / len(scores_b)
    winner = "B" if avg_b > avg_a else "A"

    return {
        "prompt_a_avg": round(avg_a, 3),
        "prompt_b_avg": round(avg_b, 3),
        "winner":       winner,
        "margin":       round(abs(avg_b - avg_a), 3),
        "n_inputs":     len(test_inputs),
    }


# Scorer: awards points for response length, question answered, polite tone
def quality_scorer(user_input: str, response: str) -> float:
    score = 0.0
    words = len(response.split())
    score += min(words / 50, 0.4)   # length (max 0.4)
    if "?" not in response:          # not just a question back
        score += 0.3
    if any(w in response.lower() for w in ["thank", "happy", "help", "support"]):
        score += 0.3
    return round(score, 2)


# Simulate LLM (in production: use openai.chat.completions)
def mock_llm(prompt: str) -> str:
    if "concise" in prompt:
        return f"Your issue has been noted. Reference: #{hash(prompt) % 10000}."
    return f"Thank you for reaching out! We are happy to help you with your request. {prompt[-30:]}"


prompt_a = "Answer this support question briefly: {input}"
prompt_b = "You are a helpful support agent. Be concise and professional. Question: {input}"

inputs = [
    "My invoice shows the wrong amount.",
    "I cannot log into my account.",
    "When will my order ship?",
    "I need a refund for last month.",
]

result = run_ab_test(prompt_a, prompt_b, inputs, mock_llm, quality_scorer)
print("A/B Test Result:")
for k, v in result.items():
    print(f"  {k}: {v}")

══════════════════════════════════════════════════════════════
 EXERCISE 5 — A/B TEST A RAG PIPELINE PROMPT
══════════════════════════════════════════════════════════════
 Use run_ab_test (defined above) to compare two RAG answer prompts.
 The test simulates what happens AFTER retrieval — the LLM formats the answer.

 Using the starting data below:
   1. Run run_ab_test with prompt_a, prompt_b, test_inputs, mock_rag_llm, and rag_scorer
   2. Print the full result dict
   3. Add a print statement: "Promote prompt B" if B wins, else "Keep prompt A"

 rag_scorer rules (implement this yourself):
   - Award 0.4 if response contains at least one number (has specific data)
   - Award 0.3 if response length > 30 words
   - Award 0.3 if response does not start with "I" (not first-person)
   - Return the sum (max 1.0)

 Expected output (scores depend on mock_rag_llm):
     A/B Test Result:
       prompt_a_avg: X.XXX
       prompt_b_avg: X.XXX
       winner: A or B
       margin: X.XXX
       n_inputs: 4
     Promote prompt B  (or: Keep prompt A)

 --- starting data ---

In [ ]:
def mock_rag_llm(prompt: str) -> str:
    context = "Q1 revenue was $4.2M. Q2 was $5.1M. Top product: Laptop (42 units)."
    if "structured" in prompt.lower():
        return f"Based on the data: {context}"
    return f"I found the following information: {context}"

prompt_a = "Answer the question using the context.\nContext: retrieved documents.\nQuestion: {input}"
prompt_b = (
    "You are a data analyst assistant. Provide a structured answer with specific numbers.\n"
    "Context: retrieved documents.\nQuestion: {input}"
)

test_inputs = [
    "What was Q1 revenue?",
    "Which product sold the most?",
    "Compare Q1 and Q2 performance.",
    "What are the top revenue drivers?",
]